# Transfer-Performance Matching
This notebook focuses on creating performance blcok/transfer value pairs,
which can ultimately be used to train a supervised Bayesian linear regression
model to predict transfer values.

In [2]:
import pandas as pd
import duckdb
from understatapi import UnderstatClient

In [3]:
# 1. Get forward data
positions = ['F']
f_stats_df = pd.read_csv(f"../data/{'_'.join(positions)}_stats.csv")

f_stats_df.head()

,player_id,date,goals_per_90,xG_per_90,assists_per_90,xA_per_90,key_passes_per_90,xGChain_per_90,xGBuildup_per_90
0,3769,2017-11-05,0.176125,0.186441,0.35225,0.123798,2.113503,0.423613,0.130932
1,3769,2018-02-18,0.200445,0.136871,0.00000,0.056769,1.603563,0.334553,0.154694
2,3769,2019-08-25,0.000000,0.264276,0.00000,0.059867,1.022727,0.437595,0.128894
3,3769,2019-12-14,0.000000,0.061819,0.00000,0.191598,1.152503,0.591171,0.429220
4,3769,2020-07-04,0.000000,0.038386,0.00000,0.066023,0.978261,0.487978,0.383570


In [6]:
# Have to get name info, not just id, to join
id_names = pd.read_csv("../data/id_names.csv")
id_names["player_id"] = id_names["player_id"].astype("int64")

f_stats_names = pd.merge(f_stats_df, id_names, on="player_id")

f_stats_names = f_stats_names.sort_values(by="date")
f_stats_names["date"] = f_stats_names['date'].astype('datetime64[us]')
f_stats_names.head()

,player_id,date,goals_per_90,xG_per_90,assists_per_90,xA_per_90,key_passes_per_90,xGChain_per_90,xGBuildup_per_90,Unnamed: 0,name
8092,4770,2014-10-17,0.242588,0.376075,0.121294,0.139135,0.606469,0.489586,0.066570,8226,Yoann Touzghar
5930,3293,2014-10-17,0.393586,0.268840,0.000000,0.130752,1.574344,0.646234,0.312876,789,Lucas Moura
10882,3294,2014-10-17,0.430622,0.511214,0.000000,0.047331,0.645933,0.525008,0.134515,1167,Edinson Cavani
11519,3309,2014-10-18,0.234375,0.143170,0.234375,0.033735,0.468750,0.292463,0.164171,7230,Valère Germain
2633,2272,2014-10-18,0.131195,0.285712,0.131195,0.147315,0.655977,0.453249,0.169004,2107,Yannick Carrasco


In [5]:
# 2. Pull forward transfer value data
con = duckdb.connect()

# We want ALL transfer data, regardless of league, for ANY player who played
# in top 5 leagues in 2021 or later
# so first need to get IDs and then filter off those

q = """
SELECT name, V.date, V.market_value_in_eur
FROM read_csv_auto('https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/players.csv.gz') P
JOIN read_csv_auto('https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/player_valuations.csv.gz') V ON P.player_id = V.player_id
WHERE P.position = 'Attack'
AND P.player_id IN (SELECT DISTINCT player_id FROM read_csv_auto('https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/player_valuations.csv.gz') WHERE player_club_domestic_competition_id IN ('GB1', 'ES1', 'GR1', 'FR1', 'IT1') AND YEAR(V.date) >= 2021)
ORDER BY V.date ASC
"""

transfer_df = con.sql(q).df()
# Already ordered by date
transfer_df["date"] = transfer_df['date'].astype('datetime64[us]')
transfer_df.head()

KeyboardInterrupt: 

In [71]:
# 3. Join transfer data with performance data - join on latest performance date that's no later than the value date
df_combined = pd.merge_asof(f_stats_names, transfer_df, on="date", by="name")

In [73]:
# Drop those with nas
df_combined = df_combined.dropna()

In [74]:
# Save to csv
df_combined.to_csv("../data/F_stats_values.csv")

In [75]:
df_combined

,player_id,date,goals_per_90,xG_per_90,assists_per_90,xA_per_90,key_passes_per_90,xGChain_per_90,xGBuildup_per_90,name,market_value_in_eur
6429,3226,2021-01-06,0.327869,0.325879,0.000000,0.184208,2.131148,0.807533,0.413355,Ousmane Dembélé,50000000.0
6432,7208,2021-01-08,0.282132,0.153102,0.282132,0.081082,1.128527,0.276931,0.042747,Samuel Chukwueze,20000000.0
6455,5212,2021-01-10,0.000000,0.224506,0.159574,0.107983,0.957447,0.359904,0.048266,Sergi Guardiola,4000000.0
6461,2589,2021-01-11,0.162162,0.364977,0.000000,0.012885,0.324324,0.446071,0.101518,Rafa Mir,5500000.0
6464,2311,2021-01-12,0.344168,0.333816,0.000000,0.076386,0.516252,0.225788,0.047312,Roberto Soldado,1200000.0
...,...,...,...,...,...,...,...,...,...,...,...
17354,2662,2026-03-15,0.000000,0.580462,0.000000,0.157141,2.058824,0.931836,0.389435,Christian Pulisic,60000000.0
17355,9275,2026-03-16,0.000000,0.266005,0.000000,0.128395,1.250000,0.502687,0.230261,Iván Romero,7500000.0
17356,9812,2026-03-16,0.260116,0.271714,0.130058,0.218489,1.820809,0.507961,0.301258,Isi Palazón,3000000.0
17357,7825,2026-03-16,0.207373,0.223177,0.207373,0.060875,1.244240,0.206852,0.059052,Iker Losada,1400000.0
